## Identity — Users vs ServiceAccounts

Two kinds of identity, easy to mix up.

**Users — for humans (and CI).** A **User** is whatever the API server derives from your kubeconfig: a client cert's `CN=alice, O=engineering` (user + group), an OIDC `sub` claim, a token file, a webhook. **Users are *not* Kubernetes objects** — there's no `kubectl create user`. The API server just *recognises* what your authenticator returns; you add one by issuing a CA-signed cert or configuring OIDC.

**ServiceAccounts — for pods.** A **ServiceAccount** *is* a namespaced API object. A pod references it by name; the kubelet mounts a **token** for it; the pod uses that token to call the API server.

```yaml
apiVersion: v1
kind: ServiceAccount
metadata: { name: api-server, namespace: default }
```

Users authenticate *people*; ServiceAccounts authenticate *workloads*. RBAC applies to either.

**The token.** The kubelet mounts three files at `/var/run/secrets/kubernetes.io/serviceaccount/`: `ca.crt`, `namespace`, `token`. Modern tokens are **bound** — ~1h expiry (auto-rotated), scoped to audiences, revoked when the pod dies. Legacy never-expiring Secret tokens are phased out (suspicious past 1.24).

Best practices: the `default` SA has **zero permissions** — make a **workload-specific SA**, don't grant `default` more than it needs; set **`automountServiceAccountToken: false`** for pods that never call the API. And the group `system:authenticated` includes *everyone* — a ClusterRoleBinding to it grants all. On our map this is the **ServiceAccount** chip in the RBAC box — the identity a Pod carries into the api-server.